# Dwell-block research

Reverse-engineering chento's **dwellblock** entry concept and testing whether it is a deployable edge.

This notebook is a re-runnable **viewer** over the study scripts in this folder; the full written record is in [`findings.md`](findings.md). Run top-to-bottom (kernel cwd = this folder). Heavy cells that rebuild triggers are marked.

**Bottom line:** the deployable edge is a *shallow ~20% limit pullback* toward the nearest dwellblock; deep "bid-the-zone" bidding (his apparent style) does not survive as a systematic edge.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.float_format', lambda v: f'{v:+.3f}')
D = Path.cwd()  # expect studies/notebooks/dwell_block
show = lambda f: display(Image(filename=str(D / f))) if (D / f).exists() else print('missing:', f)

## 1. What a dwellblock is

A horizontal price zone where price spent significant **time** consolidating (a Market-Profile / volume-profile high-time-at-price node). Reconstructed from the only 3 of his 1,989 messages that use the word (all 2026), plus their annotated charts. He uses it three ways: entry staging, support-that-holds, and breakout ("acceptance above").

### Detector validation vs his 3 actual marked charts
For L1781 the detected POCs (69,754 / 68,663) land on the 69,737 / 68,662 lines he had drawn.

In [ ]:
# regenerate: python dwellblock.py
show('dwellblock_validation.png')

## 2. Experiment 4a — pullback-depth sweep (edge signal)

Hold the signal constant (TRIPLE_V3 = B1∩B5∩B7), vary entry = pullback depth toward the nearest dwellblock POC. Expectancy **peaks at a shallow ~20% pullback** and declines to the full dwell-zone bid.

In [ ]:
# regenerate: python experiment_abc.py triple
show('depth_curve.png')

## 3. Experiment 4b — how chento ACTUALLY bids

For 140 extracted BTC trades, where does his real entry sit vs the detected zone? He bids **deep** — median ~0.9 ATR *below* the POC — the opposite side from the sweep's shallow optimum. (His deep bid is a ladder anchor; log is survivorship-skewed.)

In [ ]:
# regenerate: python analyze_chento_bids.py
show('chento_bid_depth.png')

## 4. Experiment 4c — does a ladder rescue deep bidding?

Deep entry + bounded DCA ladder + the stop-and-re-enter-next-support cascade. **No** — the ladder makes deep bidding worse (the add lands right before the stop). Shallow stays the winner.

In [ ]:
# regenerate: python experiment_ladder.py
show('ladder_compare.png')

## 5. Equity & max drawdown

**Controlled exit model** (structural stop / 3R / 72h): shallow nearly doubles return AND slightly reduces max DD; deep+ladder/cascade blow DD up 2–3×.

In [ ]:
# regenerate: python equity_compare.py
try:
    display(pd.read_csv(D / 'equity_stats.csv'))
except Exception as e:
    print('run equity_compare.py first:', e)
show('equity_curves.png')

## 6. Prod-faithful comparison (2a)

The real test: replay the **shipped chento_triple_v3 exit model** (5×ATR stop, 6R target, 72h, no ladder, prod filters) with current-market vs shallow-pullback entry. Does the shallow entry help the actual sleeve's drawdown?

In [ ]:
# regenerate: python prod_compare.py   (heavy: rebuilds triggers)
p = D / 'prod_compare_results.txt'
print(p.read_text() if p.exists() else 'run prod_compare.py first')
show('prod_compare_equity.png')

## Conclusions

1. An edge signal is mandatory (no entry trick rescues noise).
2. Deployable edge = a **shallow ~20% limit pullback** (~2× market expectancy, lower max DD), no ladder.
3. Deep "bid-the-zone" bidding does **not** survive as a systematic edge — confirmed by the depth sweep, the ladder/cascade test (which worsens DD 2–3×), and the prior backward-only ladder verdict.

**Next:** (2a) prod-faithful DD comparison (above) → if it holds, (2b) wire a dwellblock-aware shallow-pullback limit into `chento_triple_v3` (currently market entry). See [`findings.md`](findings.md) §7 for the full next-step list and caveats.